# 02 — Silver Orders Transformation

## Overview
This lab implements the **Bronze → Silver promotion layer** for orders data. It reads only the new Bronze rows since the last run, applies a chain of transformations, and uses a **Delta MERGE** to write an idempotent, deduplicated Silver table.

**What you will learn**
- How to implement an incremental watermark strategy (timestamp-based) so re-runs never double-process rows
- How to hash PII fields (SHA-256) before they leave Bronze, satisfying data minimisation requirements
- How to deduplicate late-arriving updates with a window function (keep latest `updated_at` per `order_id`)
- How to use `DeltaTable.merge()` for upsert semantics: update only if the incoming row is newer, insert otherwise
- How to write a post-merge duplicate check as a contract between layers

**Prerequisites:** Scenario 01 has been run at least once so `bronze.orders_raw` is populated.

### Cell 1 — Process-date parameter
Optional override for reprocessing a specific Bronze partition. When left empty, the notebook automatically discovers which Bronze rows have not yet been promoted to Silver by comparing timestamps — this is what makes every run **idempotent**: running the same notebook twice will produce the same Silver result, not duplicates.

In [ ]:
# ── Parameters ───────────────────────────────────────────────────────────────
p_process_date = ""   # empty = process all unprocessed partitions

### Cell 2 — Spark & Delta configuration
Same shuffle-partition and write-optimisation settings as the Bronze notebook. `DeltaTable` is imported here because the MERGE destination is a managed Delta table — `DeltaTable.forName()` gives a handle to issue `.merge()` operations programmatically via the Delta Lake Java API exposed through PySpark.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

LAKEHOUSE      = "lh_advanced_scenarios"
SOURCE_TABLE   = f"{LAKEHOUSE}.bronze.orders_raw"
TARGET_TABLE   = f"{LAKEHOUSE}.silver.orders"

### Cell 3 — Incremental watermark window
Determines which Bronze rows to process. If a target date was supplied, a simple equality filter is used. Otherwise, the notebook reads the **maximum `_silver_ts`** already written to Silver and uses it as a lower bound — only rows with a later `_ingest_ts` are included. This is the **timestamp watermark pattern**: lightweight, no extra control table needed, and safe for concurrent pipeline runs.

In [ ]:
# ── Determine processing window ──────────────────────────────────────────────
if p_process_date:
    date_filter = F.col("_ingest_date") == p_process_date
else:
    # Incremental: Bronze partitions not yet merged into Silver
    try:
        last_silver = spark.sql(
            f"SELECT MAX(_silver_ts) AS ts FROM {TARGET_TABLE}"
        ).collect()[0]["ts"]
    except Exception:
        last_silver = None

    if last_silver:
        date_filter = F.col("_ingest_ts") > F.lit(last_silver)
    else:
        date_filter = F.lit(True)

print(f"Processing filter: {date_filter}")

### Cell 4 — Read filtered Bronze rows
Reads only the qualifying Bronze rows into memory. Printing the count early is important operationally — an unexpectedly large batch may indicate the watermark was reset or a large backfill was triggered, while a zero count means the watermark filter excluded everything (normal after a re-run on already-processed data).

In [ ]:
# ── Read Bronze ──────────────────────────────────────────────────────────────
df_bronze = spark.read.format("delta").table(SOURCE_TABLE).filter(date_filter)
print(f"Bronze rows to process: {df_bronze.count():,}")

### Cell 5 — Transformations
Applies four categories of change in a single DataFrame chain:
1. **Type re-affirmation** — explicit casts ensure Silver columns are correctly typed even if Bronze drift occurs.
2. **Status normalisation** — `UPPER(TRIM(...))` eliminates casing inconsistencies (`"Shipped"` vs `"SHIPPED"`) before data reaches Silver.
3. **PII pseudonymisation** — `customer_id` is replaced with its SHA-256 hex hash (64 chars) and the raw value is dropped. This satisfies data minimisation at the storage layer.
4. **Null safety** — `COALESCE(amount, 0.00)` prevents null propagation in downstream aggregations.

In [ ]:
# ── Transformations ──────────────────────────────────────────────────────────
df_typed = (
    df_bronze
    # Type casts (already typed in schema, but re-affirm safety)
    .withColumn("order_date",       F.col("order_date").cast("date"))
    .withColumn("amount",           F.col("amount").cast("decimal(10,2)"))
    # Clean status
    .withColumn("status",           F.upper(F.trim(F.col("status"))))
    # Hash PII — SHA-256 produces 64-char hex
    .withColumn("customer_id_hash", F.sha2(F.col("customer_id"), 256))
    .drop("customer_id")                # remove raw PII
    # Default nulls
    .withColumn("amount",           F.coalesce(F.col("amount"), F.lit(0.00)))
    # Audit
    .withColumn("_silver_ts",       F.current_timestamp())
)

### Cell 6 — Deduplication with window function
Source systems often emit **multiple update events for the same order in a single batch** (e.g. a status change followed by an amount correction within the same ETL cycle). `ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC)` assigns rank 1 to the most recent row per order. Keeping only rank-1 rows ensures each `order_id` appears exactly once before the MERGE, preventing the MERGE from failing on duplicate keys in the incoming dataset.

In [ ]:
# ── Deduplicate: keep latest updated_at per order_id ─────────────────────────
w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())
df_deduped = (
    df_typed
    .withColumn("_row_num", F.row_number().over(w))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num", "_ingest_date", "_ingest_ts", "_source_file")
)

print(f"Deduplicated rows: {df_deduped.count():,}")

### Cell 7 — Delta MERGE into Silver
**First run:** creates the Silver table with a full overwrite (no previous data to merge against). **Subsequent runs:** issues a `MERGE` statement with two conditions: (a) `WHEN MATCHED` — updates the existing Silver row only if the incoming `updated_at` is strictly newer, preventing accidental downgrades from out-of-order batches; (b) `WHEN NOT MATCHED` — inserts brand-new orders. This combination makes the operation **fully idempotent**: running it twice on identical data produces identical results.

In [ ]:
# ── MERGE into Silver ────────────────────────────────────────────────────────
# Create table on first run
if not spark.catalog.tableExists(TARGET_TABLE):
    df_deduped.write.format("delta").mode("overwrite") \
              .option("overwriteSchema", "true") \
              .saveAsTable(TARGET_TABLE)
    print(f"Created {TARGET_TABLE} (first load).")
else:
    silver = DeltaTable.forName(spark, TARGET_TABLE)
    (
        silver.alias("t")
              .merge(df_deduped.alias("s"), "t.order_id = s.order_id")
              .whenMatchedUpdate(condition="s.updated_at > t.updated_at", set={
                  "order_date":       "s.order_date",
                  "updated_at":       "s.updated_at",
                  "status":           "s.status",
                  "amount":           "s.amount",
                  "customer_id_hash": "s.customer_id_hash",
                  "_silver_ts":       "s._silver_ts",
              })
              .whenNotMatchedInsertAll()
              .execute()
    )
    print(f"MERGE complete → {TARGET_TABLE}")

### Cell 8 — Post-MERGE duplicate check
Counts any `order_id` values with more than one row in Silver and asserts the result is zero. A violation here means the MERGE join condition (`t.order_id = s.order_id`) has a bug, or the deduplication step above missed something. **This assertion is a data contract**: it verifies the invariant that downstream Gold and serving queries depend on — unique orders in Silver.

In [ ]:
# ── Validation ───────────────────────────────────────────────────────────────
dups = spark.sql(
    f"SELECT order_id, COUNT(*) c FROM {TARGET_TABLE} GROUP BY order_id HAVING c > 1"
).count()
assert dups == 0, f"Duplicate order_ids found after MERGE: {dups}"

total = spark.sql(f"SELECT COUNT(*) c FROM {TARGET_TABLE}").collect()[0][0]
print(f"Silver row count: {total:,} | Duplicates: {dups}")